In [2]:
import numpy as np
import matplotlib.pyplot as plt

import gps_sim
import prn

In [3]:
f_s = 2.6e6
f_if = 100e3

samples = gps_sim.generate_gps(f_s, int(f_s*2), 1, f_if, 0, -130)

In [16]:
# Algorithm from Tsui section 7.14

x_1ms = samples[0:int(f_s*1e-3)] * prn.sample(1, f_s, int(f_s*1e-3))
x_5ms = samples[0:int(f_s*5e-3)] * prn.sample(1, f_s, int(f_s*5e-3))

# Coarse frequency
f_est = f_if + 400

# Medium frequency
t = np.arange(len(x_5ms)) / f_s
for f in (f_est + np.array([-400, 0, 400])):
    mfrq = np.abs(np.sum(x_5ms * np.exp(2j*np.pi*t*f)))
    # print(f, mfrq)

# mfrq = 100700
mfrq = f_est

# Fine freq
x_b5 = x_5ms * np.exp(-2j*np.pi*mfrq*t)
x_c5 = np.diff(-np.angle(np.sum(np.reshape(x_b5, (5, -1)), axis=1)))
x_d5 = list(x_c5)

threshold = 2.3*np.pi/5
for i in range(4):
    if abs(x_c5[i]) > threshold:
        x_c5[i] = x_d5[i] - 2*np.pi
        if abs(x_c5[i]) > threshold:
            x_c5[i] = x_d5[i] - 2*np.pi
            if abs(x_c5[i]) > 2.2*np.pi/5:
                x_c5[i] = x_d5[i] - np.pi
                if abs(x_c5[i]) > threshold:
                    x_c5[i] = x_d5[i] - 3*np.pi
                    if abs(x_c5[i]) > threshold:
                        x_c5[i] = x_d5[i] + np.pi

dfrq = np.mean(x_c5) * 1e3/(2*np.pi)
print(f_est + dfrq)

100298.20153555664
